# Model Training Sanity Check

This notebook verifies that our pipeline outputs (the Gold feature store) are machine-learning compatible and that we do not have data leakage. We'll train a simple Logistic Regression model and evaluate its performance.

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Initialize Spark Session
spark = SparkSession.builder \
    .appName('ModelTrainingCheck') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session initialized.")

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Load Gold Data
gold_df = spark.read.parquet('datamart/gold/gold_feature_store.parquet')

# Load labels from the raw loan dataset or label store (depending on implementation)
# Here we assume the Gold layer contains 'label' or we join it.
gold_df.printSchema()
gold_df.show(5)

In [ ]:
# Prepare features
# We filter out non-feature columns like IDs and dates
feature_cols = [c for c in gold_df.columns if c not in ('customer_id', 'snapshot_date', 'label')]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"  # Skip rows with nulls if any snuck through
)

data_ready = assembler.transform(gold_df).select("features", "label")

# Train/Test Split
train_df, test_df = data_ready.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train_df.count()}, Testing rows: {test_df.count()}")

In [ ]:
# Train Logistic Regression
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=10)
lr_model = lr.fit(train_df)

# Predict on test data
predictions = lr_model.transform(test_df)
predictions.select('label', 'prediction', 'probability').show(5)

In [ ]:
# Evaluate
evaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
roc_auc = evaluator.evaluate(predictions)

print(f"ROC-AUC Score: {roc_auc:.4f}")

if roc_auc > 0.99:
    print("WARNING: Extremely high ROC-AUC. Possible DATA LEAKAGE!")
elif roc_auc < 0.5:
    print("WARNING: Model performs worse than random guessing.")
else:
    print("Model performance is within normal realistic bounds.")